In [ ]:
!pip install google-cloud-aiplatform --quiet
!pip install pandas tqdm --quiet

✅ Bibliotecas instaladas


In [ ]:
from google.cloud import aiplatform
from google.cloud import storage
from google.oauth2 import service_account
import vertexai
from vertexai.preview.tuning import sft
from vertexai.generative_models import GenerativeModel
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time
import json
import os

✅ Imports completados


In [ ]:
PROJECT_ID = "kino-473215"
LOCATION = "us-central1"

DATA_PATH = Path("./data-sources/pre-processed")
TRAIN_FILE = "data_finetuning_train.csv"
VAL_FILE = "data_finetuning_val.csv"

BASE_MODEL = "gemini-2.5-flash"
TUNED_MODEL_NAME = "medical-summarizer-vertexai"

EPOCH_COUNT = 2
LEARNING_RATE = 0.0003
RANDOM_SEED = 42

MAX_INPUT_CHARS = 10000
MAX_OUTPUT_CHARS = 4000

MAX_TRAIN_SAMPLES = None 

print("Configuración cargada")
print(f"Proyecto: {PROJECT_ID}")
print(f"Región: {LOCATION}")
print(f" Modelo base: {BASE_MODEL}")
print(f"\nLímites:")
print(f" Input:  {MAX_INPUT_CHARS:,} caracteres")
print(f"Output: {MAX_OUTPUT_CHARS:,} caracteres")

✅ Configuración cargada
   Proyecto: kino-473215
   Región: us-central1
   Modelo base: gemini-2.5-flash

📊 Límites:
   Input:  10,000 caracteres
   Output: 4,000 caracteres


In [ ]:
from google.oauth2 import service_account

KEY_PATH = r"C:\Users\maaro\OneDrive\Documentos\MaestriaIA\Despliegue de soluciones\Proyecto-PLN-FLAG\src\kino-473215-6a19bca35b8a.json"

print("Cargando credenciales...")
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

SCOPES = [
    'https://www.googleapis.com/auth/cloud-platform',
    'https://www.googleapis.com/auth/devstorage.full_control',
]

credentials = credentials.with_scopes(SCOPES)

print(f"Credenciales cargadas")
print(f"Service Account: {credentials.service_account_email}")
print(f"Project ID: {credentials.project_id}")
print(f"Scopes: {len(SCOPES)} configurados")

aiplatform.init(
    project=PROJECT_ID, 
    location=LOCATION, 
    credentials=credentials
)

vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)

print(f"\nVertex AI inicializado")
print(f"Proyecto: {PROJECT_ID}")
print(f"Región: {LOCATION}")

🔐 Cargando credenciales...
✅ Credenciales cargadas
   Service Account: vertex-ai-tuning@kino-473215.iam.gserviceaccount.com
   Project ID: kino-473215
   Scopes: 2 configurados

✅ Vertex AI inicializado
   Proyecto: kino-473215
   Región: us-central1


In [ ]:
def truncate_text_by_chars(text, max_chars):
    """Trunca texto a máximo de caracteres"""
    if pd.isna(text):
        return ""
    
    text = str(text).strip()
    if len(text) <= max_chars:
        return text
    
    truncated = text[:max_chars]
    last_space = truncated.rfind(' ')
    if last_space > max_chars * 0.9:
        truncated = truncated[:last_space]
    
    return truncated


def generar_prompt(article):
    """Genera prompt para Gemini"""
    return f"""You are a medical text summarizer. Create a clear, concise Plain Language Summary of the following scientific article.

Scientific Article:
{article}

Plain Language Summary:"""


def df_to_vertex_format(df, desc="Procesando", max_samples=None):
    """
    Convierte DataFrame a formato JSONL para Vertex AI Gemini 2.5.
    
    FORMATO CORRECTO para Gemini 2.5:
    {
        "contents": [
            {"role": "user", "parts": [{"text": "prompt"}]},
            {"role": "model", "parts": [{"text": "respuesta"}]}
        ]
    }
    """
    print(f"\n{desc}...")
    print(f"   Total de filas: {len(df)}")
    
    df = df.copy()
    if max_samples and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=RANDOM_SEED).reset_index(drop=True)
        print(f"Limitado a {max_samples:,} muestras")
    
    examples = []
    skipped = 0
    truncated_inputs = 0
    truncated_outputs = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  {desc}"):
        article = str(row['article']).strip()
        summary = str(row['summary']).strip()
        
        # Validar no vacíos
        if not article or not summary or article == 'nan' or summary == 'nan':
            skipped += 1
            continue
        
        # Validar longitud mínima
        if len(article) < 50 or len(summary) < 20:
            skipped += 1
            continue
        
        # Truncar por caracteres
        original_article_len = len(article)
        original_summary_len = len(summary)
        
        article = truncate_text_by_chars(article, MAX_INPUT_CHARS)
        summary = truncate_text_by_chars(summary, MAX_OUTPUT_CHARS)
        
        if original_article_len > MAX_INPUT_CHARS:
            truncated_inputs += 1
        if original_summary_len > MAX_OUTPUT_CHARS:
            truncated_outputs += 1
        
        # Crear prompt
        prompt = generar_prompt(article)
        
        example = {
            "contents": [
                {
                    "role": "user",
                    "parts": [{"text": prompt}]
                },
                {
                    "role": "model",
                    "parts": [{"text": summary}]
                }
            ]
        }
        examples.append(example)
    
    print(f"Ejemplos procesados: {len(examples):,}")
    if skipped > 0:
        print(f"Omitidos: {skipped} (vacíos o muy cortos)")
    if truncated_inputs > 0:
        print(f"Inputs truncados: {truncated_inputs} (>{MAX_INPUT_CHARS:,} chars)")
    if truncated_outputs > 0:
        print(f"Outputs truncados: {truncated_outputs} (>{MAX_OUTPUT_CHARS:,} chars)")
    
    return examples

print("Funciones de procesamiento definidas")
print("Formato: GenerateContent (Gemini 2.5)")

✅ Funciones de procesamiento definidas
   Formato: GenerateContent (Gemini 2.5)


In [ ]:
print("="*80)
print("CARGANDO DATOS")
print("="*80)

train_path = DATA_PATH / TRAIN_FILE
val_path = DATA_PATH / VAL_FILE

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

print(f"\nDatasets cargados:")
print(f"Train: {len(train_df):,} ejemplos")
print(f"Val:   {len(val_df):,} ejemplos")

train_data = df_to_vertex_format(train_df, "TRAIN", MAX_TRAIN_SAMPLES)
val_data = df_to_vertex_format(val_df, "VAL")

print(f"\n{'='*80}")
print("DATOS PREPARADOS")
print("="*80)
print(f"Train: {len(train_data):,} ejemplos listos")
print(f"Val:   {len(val_data):,} ejemplos listos")

📂 CARGANDO DATOS

✅ Datasets cargados:
   Train: 3,038 ejemplos
   Val:   380 ejemplos

🔄 TRAIN...
   Total de filas: 3038


  TRAIN: 100%|██████████| 3038/3038 [00:00<00:00, 24752.94it/s]


   ✅ Ejemplos procesados: 3,038
   ✂️  Inputs truncados: 3 (>10,000 chars)
   ✂️  Outputs truncados: 553 (>4,000 chars)

🔄 VAL...
   Total de filas: 380


  VAL: 100%|██████████| 380/380 [00:00<00:00, 27790.89it/s]

   ✅ Ejemplos procesados: 380
   ✂️  Outputs truncados: 57 (>4,000 chars)

✅ DATOS PREPARADOS
   Train: 3,038 ejemplos listos
   Val:   380 ejemplos listos


In [ ]:
train_jsonl_path = "train_data.jsonl"
val_jsonl_path = "val_data.jsonl"

with open(train_jsonl_path, 'w', encoding='utf-8') as f:
    for example in train_data:
        f.write(json.dumps(example, ensure_ascii=False) + '\n')

with open(val_jsonl_path, 'w', encoding='utf-8') as f:
    for example in val_data:
        f.write(json.dumps(example, ensure_ascii=False) + '\n')

print("Datos guardados en formato JSONL")
print(f"Train: {train_jsonl_path}")
print(f"Val:   {val_jsonl_path}")

✅ Datos guardados en formato JSONL
   Train: train_data.jsonl
   Val:   val_data.jsonl


In [ ]:
BUCKET_NAME = f"{PROJECT_ID}-gemini-tuning"
BUCKET_URI = f"gs://{BUCKET_NAME}"

storage_client = storage.Client(
    project=PROJECT_ID,
    credentials=credentials
)

# Crear bucket si no existe
try:
    bucket = storage_client.create_bucket(
        BUCKET_NAME, 
        location=LOCATION
    )
    print(f"Bucket creado: {BUCKET_NAME}")
except Exception as e:
    if "already exists" in str(e).lower() or "you already own" in str(e).lower():
        # Bucket ya existe
        bucket = storage_client.bucket(BUCKET_NAME)
        print(f"Usando bucket existente: {BUCKET_NAME}")
    else:
        raise e

print("\nSubiendo archivos a Cloud Storage...")

blob_train = bucket.blob("train_data.jsonl")
blob_train.upload_from_filename(train_jsonl_path)
print(f"Train subido")

blob_val = bucket.blob("val_data.jsonl")
blob_val.upload_from_filename(val_jsonl_path)
print(f" Validation subido")

TRAIN_URI = f"{BUCKET_URI}/train_data.jsonl"
VAL_URI = f"{BUCKET_URI}/val_data.jsonl"

print(f"\nDatos subidos a Cloud Storage")
print(f"Train: {TRAIN_URI}")
print(f"Val:   {VAL_URI}")

✅ Usando bucket existente: kino-473215-gemini-tuning

⬆️  Subiendo archivos a Cloud Storage...
   ✅ Train subido
   ✅ Validation subido

✅ Datos subidos a Cloud Storage
   Train: gs://kino-473215-gemini-tuning/train_data.jsonl
   Val:   gs://kino-473215-gemini-tuning/val_data.jsonl


In [ ]:
print("="*80)
print("INICIANDO FINE-TUNING CON VERTEX AI")
print("="*80)

print(f"\nConfiguración:")
print(f"   Modelo base:    {BASE_MODEL}")
print(f"   Nombre:         {TUNED_MODEL_NAME}")
print(f"   Épocas:         {EPOCH_COUNT}")
print(f"   Learning rate:  {LEARNING_RATE}")
print(f"   Train data:     {TRAIN_URI}")
print(f"   Val data:       {VAL_URI}")

print(f"\n⏳ Creando trabajo de tuning...")

from vertexai.preview.tuning import sft

sft_tuning_job = sft.train(
    source_model=BASE_MODEL,
    train_dataset=TRAIN_URI,
    validation_dataset=VAL_URI,
    epochs=EPOCH_COUNT,
    learning_rate_multiplier=LEARNING_RATE,
    tuned_model_display_name=TUNED_MODEL_NAME,
)

print(f"\nTrabajo de tuning creado!")
print(f"   ID: {sft_tuning_job.resource_name}")
print(f"   Estado: {sft_tuning_job.state}")

🚀 INICIANDO FINE-TUNING CON VERTEX AI

⚙️  Configuración:
   Modelo base:    gemini-2.5-flash
   Nombre:         medical-summarizer-vertexai
   Épocas:         2
   Learning rate:  0.0003
   Train data:     gs://kino-473215-gemini-tuning/train_data.jsonl
   Val data:       gs://kino-473215-gemini-tuning/val_data.jsonl

⏳ Creando trabajo de tuning...
Creating SupervisedTuningJob
SupervisedTuningJob created. Resource name: projects/270545596289/locations/us-central1/tuningJobs/1329970312929869824
To use this SupervisedTuningJob in another session:
tuning_job = sft.SupervisedTuningJob('projects/270545596289/locations/us-central1/tuningJobs/1329970312929869824')
View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/1329970312929869824?project=270545596289



✅ Trabajo de tuning creado!
   ID: projects/270545596289/locations/us-central1/tuningJobs/1329970312929869824
   Estado: 2


In [ ]:
print(f"\nMonitoreando entrenamiento...")
print(f"   Esto puede tomar 1-2 horas")
print(f"   Puedes cerrar este notebook y volver después\n")

print("Refrescando estado cada 60 segundos...")

import time
while sft_tuning_job.state.name not in ["JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED"]:
    time.sleep(60)
    sft_tuning_job.refresh()
    print(f"   Estado: {sft_tuning_job.state.name}...")

print(f"\n{'='*80}")
if sft_tuning_job.state.name == "JOB_STATE_SUCCEEDED":
    print("¡ENTRENAMIENTO COMPLETADO EXITOSAMENTE!")
else:
    print(f"Entrenamiento terminó con estado: {sft_tuning_job.state.name}")
print("="*80)

if sft_tuning_job.state.name == "JOB_STATE_SUCCEEDED":
    tuned_model_name = sft_tuning_job.tuned_model_name
    tuned_model_endpoint = sft_tuning_job.tuned_model_endpoint_name
    print(f"\nModelo disponible:")
    print(f"   Nombre: {tuned_model_name}")
    print(f"   Endpoint: {tuned_model_endpoint}")


⏳ Monitoreando entrenamiento...
   Esto puede tomar 1-2 horas
   Puedes cerrar este notebook y volver después

Refrescando estado cada 60 segundos...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...


   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_RUNNING...
   Estado: JOB_STATE_SUCCEEDED...

✅ ¡ENTRENAMIENTO COMPLETADO EXITOSAMENTE!

📦 Modelo disponible:
   No

In [ ]:
print("="*80)
print("INFORMACIÓN DEL MODELO ENTRENADO")
print("="*80)

if sft_tuning_job.state.name == "JOB_STATE_SUCCEEDED":
    print(f"\nModelo entrenado exitosamente")
    print(f"\nInformación del modelo:")
    print(f"   Nombre del modelo: {sft_tuning_job.tuned_model_name}")
    print(f"   Endpoint: {sft_tuning_job.tuned_model_endpoint_name}")
    
    print(f"\n💡 Cómo usar tu modelo entrenado:")
    print(f"\n1. Desde Python:")
    print(f"""
from vertexai.preview.generative_models import GenerativeModel

# Usar el modelo fine-tuned
model = GenerativeModel("{sft_tuning_job.tuned_model_name}")

# Generar contenido
response = model.generate_content("Your prompt here")
print(response.text)
""")
    
    print(f"\n2. Desde API:")
    print(f"""
Endpoint: {sft_tuning_job.tuned_model_endpoint_name}
Modelo: {sft_tuning_job.tuned_model_name}
""")
    
    print(f"\nPrueba rápida:")
    print(f"   Para probar el modelo, crea una nueva celda con:")
    print(f"""
from vertexai.preview.generative_models import GenerativeModel

# Cargar modelo
tuned_model = GenerativeModel("{sft_tuning_job.tuned_model_name}")

# Texto de prueba
test_text = "Your medical article here..."

# Generar resumen
response = tuned_model.generate_content(test_text)
print(response.text)
""")
    
else:
    print(f"\nEl modelo no se entrenó correctamente")
    print(f"   Estado: {sft_tuning_job.state.name}")
    
print("\n" + "="*80)

🧪 INFORMACIÓN DEL MODELO ENTRENADO

✅ Modelo entrenado exitosamente

📦 Información del modelo:
   Nombre del modelo: projects/270545596289/locations/us-central1/models/1519607532958515200@1
   Endpoint: projects/270545596289/locations/us-central1/endpoints/6135951082641162240

💡 Cómo usar tu modelo entrenado:

1. Desde Python:

from vertexai.preview.generative_models import GenerativeModel

# Usar el modelo fine-tuned
model = GenerativeModel("projects/270545596289/locations/us-central1/models/1519607532958515200@1")

# Generar contenido
response = model.generate_content("Your prompt here")
print(response.text)


2. Desde API:

Endpoint: projects/270545596289/locations/us-central1/endpoints/6135951082641162240
Modelo: projects/270545596289/locations/us-central1/models/1519607532958515200@1


📝 Prueba rápida:
   Para probar el modelo, crea una nueva celda con:

from vertexai.preview.generative_models import GenerativeModel

# Cargar modelo
tuned_model = GenerativeModel("projects/27054559